# Search-03g · L'axe mémoire : ce qu'A\* garde en frontière, ce qu'IDA\* garde en pile

Ce carnet est une **accrétion de [Search-03 (recherche informée et A\*)](Search-03-Informed.ipynb)**, dans la même veine que 03b à 03f. Il prolonge aussi [Search-03f (réparer localement sous garantie)](Search-03f-Reparer-Localement-Sous-Garantie-Python.ipynb), qui mesurait le **coût de réparation** d'un plan ; ici on mesure le **coût de la mémoire**.

**Navigation** : [<< Search-03f](Search-03f-Reparer-Localement-Sous-Garantie-Python.ipynb) | [Index](README.md) | [Search-03 (track principal)](Search-03-Informed.ipynb)

**Kernel** : Python 3

***

## Le trou que ce carnet comble

La série mesure partout les **expansions** : Search-03 compare A\* et IDA\* sur ce seul axe (« IDA\* explore 2,7 à 9,6 fois plus que A\* »). Elle parle de mémoire **en prose** — `O(b^d)`, « mémoire limitée » — et ne la compte **nulle part**. Un lecteur qui veut savoir *combien* de mémoire A\* consomme sur une instance donnée, ou *pourquoi* IDA\* tient dans une pile minuscule, ne trouve aucun chiffre dans la série.

Ce carnet compte trois choses que personne ne comptait :

| Ce qui est compté | Comment | Pourquoi ce n'est pas la même chose |
|---|---|---|
| **Pic de frontière** d'A\* | taille maximale de la file de priorité | c'est la mémoire vivante de l'algorithme, celle qui explose en `O(b^d)` |
| **Taille du dictionnaire d'états** d'A\* | nombre d'états atteints (`best_g`) | la frontière seule sous-estime la mémoire réelle d'un A\* en graphe |
| **Pic de pile** d'IDA\* | profondeur maximale atteinte par la récursion | ce n'est pas une frontière : c'est la longueur du chemin, donc `O(d)` |

Et un **contrôle discriminant** : un A\* dont la frontière est **plafonnée** à `B` entrées. Au-delà du budget, on jette le pire `f` — exactement ce que fait un algorithme à mémoire bornée, sauf que celui-ci ne rattrape rien. Le carnet mesure ce que cela coûte à la garantie d'optimalité, avec un témoin positif et un témoin négatif **sur la même instance**.


## 1. Le constat, mesuré

Avant d'affirmer que la série ne mesure pas la mémoire, on le vérifie. Le contrôle porte sur les algorithmes à mémoire bornée, qui sont précisément ceux qu'un carnet sur la mémoire devrait nommer :

```
$ grep -rn "SMA\*\|SMA_star\|SMA-Star" MyIA.AI.Notebooks/Search/
```

**0 occurrence.** Les carnets 03/03d/03e écrivent `O(b^d)` dans une cellule markdown et passent. Ce carnet-ci ne comble pas le manque en citant SMA\* de loin : il **mesure** d'abord ce qu'un plafond de frontière fait à l'optimalité, puis nomme SMA\* en §6 comme l'algorithme qui, lui, garde les deux propriétés — sans l'implémenter, car une implémentation non vérifiée ne vaut pas mieux qu'une absence.


## 2. Le monde : le taquin 8

Le taquin 8 (grille 3×3, 8 tuiles et une case vide) est le terrain canonique de la littérature sur la mémoire bornée : assez grand pour que la frontière d'A\* dépasse la centaine d'entrées, assez petit pour qu'un A\* exhaustif reste instantané, et d'une **difficulté réglable** — c'est exactement ce qu'il faut pour montrer que le pic de frontière n'est pas une constante du problème.

L'état est un tuple de 9 entiers, `0` désignant la case vide. Les voisins sont les déplacements légaux de la case vide, **triés** : l'ordre d'itération fait partie des conditions de reproductibilité, un ensemble non ordonné rendrait les compteurs non reproductibles d'une session de noyau à l'autre.


In [1]:
from math import inf
import heapq
import random

GOAL = (1, 2, 3, 4, 5, 6, 7, 8, 0)
SIZE = 3
MOVES = ((-1, 0), (1, 0), (0, -1), (0, 1))


def neighbors(state):
    """Voisins legaux de la case vide, tries (determinisme des compteurs)."""
    i = state.index(0)
    r, c = divmod(i, SIZE)
    out = []
    for dr, dc in MOVES:
        nr, nc = r + dr, c + dc
        if 0 <= nr < SIZE and 0 <= nc < SIZE:
            j = nr * SIZE + nc
            lst = list(state)
            lst[i], lst[j] = lst[j], lst[i]
            out.append(tuple(lst))
    return sorted(out)


def manhattan(state):
    """Distance de Manhattan : admissible et coherente sur le taquin."""
    d = 0
    for idx, v in enumerate(state):
        if v == 0:
            continue
        r, c = divmod(idx, SIZE)
        gr, gc = divmod(v - 1, SIZE)
        d += abs(r - gr) + abs(c - gc)
    return d


def render(state):
    rows = []
    for r in range(SIZE):
        rows.append(" ".join("." if v == 0 else str(v) for v in state[r * SIZE:(r + 1) * SIZE]))
    return " / ".join(rows)


print("etat but :", render(GOAL))
print("voisins de l'etat but :", len(neighbors(GOAL)), "| h(but) =", manhattan(GOAL))


etat but : 1 2 3 / 4 5 6 / 7 8 .
voisins de l'etat but : 2 | h(but) = 0


## 3. Trois instances, une difficulté réglable

On fabrique les instances par **marche aléatoire depuis l'état but** — donc toutes sont résolubles par construction, sans avoir à tester la parité des inversions. La graine est fixée : les trois instances sont les mêmes à chaque exécution.

Les longueurs de marche sont choisies pour couvrir trois régimes : une instance où A\* n'ouvre presque rien, une intermédiaire, et une où la frontière devient le facteur limitant.


In [2]:
def scramble(n_moves, seed):
    """Marche aleatoire depuis le but : l'instance est resoluble par construction."""
    rng = random.Random(seed)
    state, prev = GOAL, None
    for _ in range(n_moves):
        cands = [s for s in neighbors(state) if prev is None or s != prev]
        prev, state = state, rng.choice(cands)
    return state


INSTANCES = [
    ("facile-1", scramble(4, 1)),
    ("facile-2", scramble(8, 2)),
    ("difficile", scramble(60, 7)),
]
for name, s in INSTANCES:
    print(f"{name:10} {render(s)}   h = {manhattan(s)}")


facile-1   . 1 2 / 4 5 3 / 7 8 6   h = 4
facile-2   1 5 2 / 7 . 3 / 8 4 6   h = 8
difficile  1 2 5 / 7 4 3 / . 6 8   h = 8


## 4. A\* : la mémoire est le pic de frontière

Un A\* en **graphe** ne peut pas se contenter d'une file : il lui faut aussi se souvenir du meilleur `g` atteint pour chaque état, sinon il ré-ouvre indéfiniment les mêmes nœuds. La mémoire vivante de l'algorithme est donc double, et c'est la seconde partie qu'on oublie :

- le **pic de frontière** — la taille maximale de la file de priorité ;
- la **taille du dictionnaire d'états** — le nombre d'états distincts atteints.

Le protocole de mesure est explicite : on échantillonne la taille de la file **après chaque expansion**, et on retient le maximum. Les ex æquo de `f` sont départagés par un **compteur d'insertion** monotone, ce qui rend la recherche entièrement déterministe — sans lui, l'ordre de `heapq` entre deux états de même `f` dépendrait de leur représentation, et les compteurs changeraient d'une exécution à l'autre.


In [3]:
class BudgetExceeded(Exception):
    pass


def astar(start, cap=None, max_expansions=200_000):
    """A* en graphe. cap=None : A* canonique.

    Mesures :
      pic_frontiere : taille maximale de la file de priorite (apres chaque expansion)
      etats         : nombre d'etats distincts atteints (taille de best_g)
      evictions     : nombre d'entrees jetees quand cap est franchi
    """
    if start == GOAL:
        return {"cost": 0, "expansions": 0, "pic_frontiere": 0, "etats": 1, "evictions": 0}
    counter = 0
    heap = [(manhattan(start), 0, counter, start)]
    counter += 1
    best_g = {start: 0}
    pic = 1
    expansions = 0
    evictions = 0
    while heap:
        f, g, _c, state = heapq.heappop(heap)
        if state == GOAL:
            return {"cost": g, "expansions": expansions, "pic_frontiere": pic,
                    "etats": len(best_g), "evictions": evictions}
        if g > best_g.get(state, inf):
            continue
        expansions += 1
        if expansions > max_expansions:
            raise BudgetExceeded("plafond d'expansions atteint")
        for nb in neighbors(state):
            ng = g + 1
            if ng < best_g.get(nb, inf):
                best_g[nb] = ng
                heapq.heappush(heap, (ng + manhattan(nb), ng, counter, nb))
                counter += 1
        if cap is not None and len(heap) > cap:
            # jeter le pire f de la frontiere (a f egal, le plus profond)
            worst = max(range(len(heap)), key=lambda k: (heap[k][0], heap[k][1]))
            victim = heap[worst]
            heap[worst] = heap[-1]
            heap.pop()
            if worst < len(heap):
                heapq.heapify(heap)
            evictions += 1
            if victim[3] in best_g:
                del best_g[victim[3]]
        pic = max(pic, len(heap))
    raise BudgetExceeded("frontiere vide sans atteindre le but")


print(f"{'instance':10} {'cout':>5} {'expansions':>11} {'pic frontiere':>14} {'etats':>7}")
for name, s in INSTANCES:
    r = astar(s)
    print(f"{name:10} {r['cost']:>5} {r['expansions']:>11} {r['pic_frontiere']:>14} {r['etats']:>7}")


instance    cout  expansions  pic frontiere   etats
facile-1       4           4              4       8
facile-2       8          12             12      24
difficile     16         219            135     354


### Lecture du résultat

Trois choses, toutes mesurées et aucune affirmée :

1. **Le pic de frontière dépend de l'instance, pas du problème.** 4 entrées sur `facile-1`, 12 sur `facile-2`, **135** sur `difficile` : plus de trente fois plus, pour un problème identique. C'est la raison pour laquelle « la mémoire d'A\* » n'a pas de valeur unique — un carnet qui écrirait `O(b^d)` sans instance dit vrai et n'apprend rien.
2. **Le pic de frontière sous-estime la mémoire réelle.** Le dictionnaire d'états vaut 8, 24 puis **354** — toujours plus gros que la file (4, 12, 135), et il ne se vide jamais : c'est lui qui borne la mémoire d'un A\* en graphe.
3. **Le coût ne bouge pas avec la mémoire** : 4, 8 et 16, c'est-à-dire l'optimum, parce qu'une frontière non bornée ne jette rien. C'est le point de comparaison du §6.


## 5. IDA\* : la mémoire tient dans une pile, le temps paie

IDA\* (Korf, 1985) échange la frontière contre une **récursion** : à chaque itération, une recherche en profondeur bornée par un seuil `f ≤ seuil`, et le seuil remonte au plus petit `f` dépassé. Sa mémoire est donc **linéaire en profondeur**, `O(b·d)` — `d` cadres de récursion, chacun gardant l'itérateur sur ses successeurs. Ce que le code mesure ici est le **nombre de cadres effectivement atteint**, c'est-à-dire `d` lui-même.

**Les deux compteurs ne mesurent pas la même chose, et le carnet le dit** : le « pic de pile » d'IDA\* n'est pas comparable terme à terme au « pic de frontière » d'A\*. L'un compte des **entrées de file** (des états en attente d'être développés), l'autre compte des **cadres de récursion** (des états sur le chemin courant). Ce sont deux monnaies de mémoire différentes ; les mettre côte à côte dit quelque chose de leur ordre de grandeur, pas une égalité d'unités.


In [4]:
def ida_star(start, max_expansions=5_000_000):
    """IDA* : DFS borne sur f. Mesure le pic de pile (= profondeur atteinte) et les expansions."""
    seuil = manhattan(start)
    pic_pile = 0
    expansions = 0

    def dfs(state, g, seuil):
        nonlocal pic_pile, expansions
        f = g + manhattan(state)
        if f > seuil:
            return f, None
        if state == GOAL:
            return "TROUVE", g
        pic_pile = max(pic_pile, g)
        expansions += 1
        if expansions > max_expansions:
            raise BudgetExceeded("plafond d'expansions atteint")
        suivant = inf
        for nb in neighbors(state):
            r, cost = dfs(nb, g + 1, seuil)
            if r == "TROUVE":
                return "TROUVE", cost
            if r < suivant:
                suivant = r
        return suivant, None

    while True:
        r, cost = dfs(start, 0, seuil)
        if r == "TROUVE":
            return {"cost": cost, "expansions": expansions, "pic_pile": pic_pile, "seuil": seuil}
        if r == inf:
            return {"cost": None, "expansions": expansions, "pic_pile": pic_pile, "seuil": seuil}
        seuil = r


print(f"{'instance':10} {'cout A*':>8} {'cout IDA*':>10} {'exp A*':>8} {'exp IDA*':>10} {'pic front.':>11} {'pic pile':>9}")
for name, s in INSTANCES:
    a = astar(s)
    i = ida_star(s)
    print(f"{name:10} {a['cost']:>8} {i['cost']:>10} {a['expansions']:>8} {i['expansions']:>10} {a['pic_frontiere']:>11} {i['pic_pile']:>9}")


instance    cout A*  cout IDA*   exp A*   exp IDA*  pic front.  pic pile
facile-1          4          4        4          4           4         3
facile-2          8          8       12         12          12         7


difficile        16         16      219      18572         135        15


### Lecture du résultat

Le même coût, deux monnaies différentes.

**L'optimalité est intacte** : IDA\* rend 4, 8 et 16, exactement les coûts d'A\*. Elle est préservée par construction, puisque la borne `f ≤ seuil` ne coupe qu'un chemin qui ne peut pas battre le coût courant.

**Le prix se paie en expansions** : 4, 12 puis **18 572** sur l'instance difficile, contre 219 pour A\* — un facteur ≈ **85**. Chaque itération re-développe ce que la précédente avait déjà ouvert, et ce que l'on économise en mémoire se repaie en temps.

**Le pic de pile suit le chemin, pas la frontière** : 3, 7 puis 15 cadres pour des coûts de 4, 8 et 16 (soit `coût − 1`, la case but n'étant pas développée), là où la frontière d'A\* valait 4, 12 et 135. Les deux séries ne sont pas proportionnelles : c'est le vrai argument d'IDA\*, et il est visible ici sans être supposé.

Une réserve d'honnêteté : le rapport d'expansions mesuré ici (≈85×) n'est **pas** celui de Search-03 (2,7 à 9,6×). Ce n'est pas une contradiction — Search-03 mesure sur un **graphe routier** (Rennes→Nice et autres trajets) avec un autre compteur ; ici le terrain est un taquin 8 et la profondeur d'itération joue un rôle bien plus brutal. Les deux mesures disent la même chose à des échelles différentes : le rapport dépend du terrain. Ce carnet ne prétend pas réconcilier les deux chiffres.


## 6. Contrôle discriminant : ce qu'un plafond de frontière fait à la garantie

Voici le contrôle qui justifie tout le carnet. On reprend le **même** A\*, avec une seule différence : quand la frontière dépasse `B` entrées, on **jette** le pire `f`. C'est le geste minimal d'un algorithme à mémoire bornée — sauf que celui-ci ne se souvient de rien : le nœud jeté est perdu, et l'optimalité avec lui.

Le protocole est un balayage : la même instance difficile, quatre budgets `B ∈ {8, 32, 64, 128}`, plus les deux instances faciles en contrôle. Chaque ligne porte son coût et le nombre d'évictions. Le témoin **positif** est le budget assez large (coût = optimum d'A\*) ; le témoin **négatif** est le budget trop serré (coût **strictement** supérieur) — et les deux doivent tomber sur la **même** instance, sans quoi le défaut serait un artefact de terrain.


In [5]:
BUDGETS = (8, 32, 64, 128)

resultats = {}
for name, s in INSTANCES:
    opt = astar(s)["cost"]
    resultats[name] = {"opt": opt, "rows": []}
    for B in BUDGETS:
        try:
            r = astar(s, cap=B)
            resultats[name]["rows"].append((B, r["cost"], r["expansions"], r["evictions"], r["etats"]))
        except BudgetExceeded as e:
            resultats[name]["rows"].append((B, None, None, None, None))

for name, s in INSTANCES:
    opt = resultats[name]["opt"]
    print(f"--- {name} (optimum A* = {opt}) ---")
    print(f"{'B':>4} {'cout':>6} {'expansions':>11} {'evictions':>10} {'etats':>7} {'verdict':>22}")
    for B, cost, exp, ev, etats in resultats[name]["rows"]:
        if cost is None:
            verdict = "frontiere epuisee"
        elif cost == opt:
            verdict = "optimal (temoin +)"
        else:
            verdict = f"SOUS-OPTIMAL (+{cost - opt})"
        print(f"{B:>4} {str(cost):>6} {str(exp):>11} {str(ev):>10} {str(etats):>7} {verdict:>22}")
    print()


--- facile-1 (optimum A* = 4) ---
   B   cout  expansions  evictions   etats                verdict
   8      4           4          0       8     optimal (temoin +)
  32      4           4          0       8     optimal (temoin +)
  64      4           4          0       8     optimal (temoin +)
 128      4           4          0       8     optimal (temoin +)

--- facile-2 (optimum A* = 8) ---
   B   cout  expansions  evictions   etats                verdict
   8      8          12          4      20     optimal (temoin +)
  32      8          12          0      24     optimal (temoin +)
  64      8          12          0      24     optimal (temoin +)
 128      8          12          0      24     optimal (temoin +)

--- difficile (optimum A* = 16) ---
   B   cout  expansions  evictions   etats                verdict
   8     32         210        145     219     SOUS-OPTIMAL (+16)
  32     32         703        434     735     SOUS-OPTIMAL (+16)
  64     16         215         70  

### Lecture du résultat

Le contrôle porte ses deux témoins, **sur la même instance** (`difficile`, optimum 16) :

- **témoin positif** — à `B = 64` et `B = 128`, le coût revient **exactement** à 16, l'optimum d'A\*. Le plafond n'est pas un sabotage : il élague une frontière qui n'avait pas besoin d'être si large.
- **témoin négatif** — à `B = 8` et `B = 32`, le coût monte à **32**, soit **+16**, le double de l'optimum. La frontière élaguée a jeté le nœud qui portait la solution optimale, et rien ne le signale à l'appelant.
- **les deux instances faciles restent optimales même au plus petit budget** (4 et 8, y compris à `B = 8` où `facile-2` subit pourtant 4 évictions) : le défaut n'est donc pas fabriqué par le protocole. Il apparaît quand le budget passe **sous le besoin réel** de l'instance.

**Ce que le pic du §4 ne dit pas — et c'est le résultat le plus intéressant.** On pourrait croire que le budget devient dangereux dès qu'il passe sous le pic mesuré (135). C'est faux, et la mesure le montre : à `B = 64`, bien en dessous de 135, le coût reste optimal. Le pic est un **majorant transitoire**, pas un seuil de rupture ; ce qui compte n'est pas la taille atteinte par la frontière mais **quel** nœud est jeté. À `B = 64`, 70 évictions ont eu lieu sans emporter le nœud critique ; à `B = 32`, 434 évictions l'ont emporté. Le seuil de mensonge se situe donc **entre 32 et 64**, et il ne se lit pas dans le pic : il faudrait le mesurer instance par instance.

Conséquence pratique, celle qui vaut pour un vrai système : un A\* à frontière plafonnée rend un plan dont l'appelant **ne peut pas savoir** qu'il est sous-optimal. Le coût n'est pas faux — il est le coût du plan rendu — mais il n'est plus garanti.


## 7. Pourquoi SMA\* n'est pas mesuré ici

L'algorithme qui garde **les deux** propriétés — frontière bornée *et* optimalité préservée — est **SMA\*** (Russell, 1992, *Efficient memory-bounded search methods*, ECAI-92). Il ne jette rien : il oublie le pire nœud de la frontière **en mémorisant son `f` sur son ancêtre**, de sorte que le sous-arbre abandonné puisse être régénéré si le budget se libère, et que la recherche reste complète et optimale.

Ce carnet le **nomme** et **ne l'implémente pas**, volontairement. Une implémentation non vérifiée de SMA\* — la propagation du `f` mémorisé sur l'ancêtre est précisément l'endroit où l'on se trompe — produirait un chiffre que personne ne pourrait contrôler, et ce chiffre aurait l'autorité d'une mesure. Le contrôle du §6 établit ce que SMA\* doit réparer ; l'écrire est un travail séparé, qui mérite son propre témoin.

> Russell, S. (1992). *Efficient memory-bounded search methods*. ECAI-92. — référence **citée**, non archivée dans le gisement partagé à ce jour : le carnet s'appuie sur la description de l'algorithme et ne lui attribue aucun chiffre.


## 8. Synthèse

| Question | Réponse mesurée dans ce carnet |
|---|---|
| La mémoire d'A\* a-t-elle une valeur ? | Non : pic de frontière 4 / 12 / **135** et dictionnaire 8 / 24 / **354** sur trois instances du même problème |
| Le pic de frontière suffit-il à décrire la mémoire d'A\* ? | Non : le dictionnaire d'états est 2 à 3 fois plus gros et ne se vide jamais |
| IDA\* économise-t-il la mémoire ? | Oui, structurellement (`O(b·d)` au lieu de `O(b^d)`) : 3 / 7 / **15** cadres contre 4 / 12 / **135** entrées de file |
| À quel prix ? | En expansions : 219 → **18 572** (≈ 85×) sur l'instance difficile |
| Un plafond de frontière préserve-t-il l'optimalité ? | Non : 16 → **32** à `B ≤ 32`, deux témoins sur la même instance |
| Le pic mesuré prédit-il le seuil de rupture ? | **Non** : à `B = 64` (< 135) le coût reste optimal ; le seuil est entre 32 et 64 et ne se lit pas dans le pic |
| Que répare SMA\* ? | Exactement cela : il oublie sans perdre, en mémorisant le `f` sur l'ancêtre — nommé, non mesuré ici |

Le carnet n'ajoute aucun algorithme à la série ; il ajoute l'**axe** qui manquait, avec les compteurs qui le rendent falsifiable.


### Exercice 1 — Prédire le pic avant de le mesurer

Sans exécuter A\*, prédisez le pic de frontière d'une instance à `k` coups du but, puis comparez à la mesure sur `k ∈ {2, 4, 6, 8}`. Le résultat n'est pas le pic lui-même : c'est **l'écart** entre votre prédiction et la mesure, et ce qu'il vous apprend sur ce que la frontière contient réellement à chaque étape.


In [6]:
def exercice_1_pic_predit(k):
    # TODO etudiant : renvoyer le pic de frontiere predit pour une instance a k coups du but,
    # puis comparer a astar(scramble(k, seed))['pic_frontiere'] sur plusieurs graines.
    print("Exercice 1 a completer : predictez le pic de frontiere avant de le mesurer")
    return None


print("Exercice 1 a completer : predictez le pic de frontiere avant de le mesurer")


Exercice 1 a completer : predictez le pic de frontiere avant de le mesurer


### Exercice 2 — Un plafond qui se souvient

Le plafond du §6 jette le pire `f` et l'oublie. Modifiez-le pour **mémoriser** ce `f` sur le parent du nœud jeté, et observez si le coût redevient optimal à budget égal. C'est le premier pas de SMA\* ; le témoin à conserver est le balayage du §6 tel quel — même instance, mêmes budgets.


In [7]:
def astar_plafond_memorisant(start, cap):
    # TODO etudiant : comme astar(cap=...), mais en memorisant le f du noeud jete sur son parent.
    # Mesurez ensuite le meme balayage de budgets et comparez au tableau du §6.
    print("Exercice 2 a completer : plafond memorisant (premier pas vers SMA*)")
    return None


print("Exercice 2 a completer : plafond memorisant (premier pas vers SMA*)")


Exercice 2 a completer : plafond memorisant (premier pas vers SMA*)


### Exercice 3 — Le prix d'IDA\* selon l'heuristique

Rejouez la comparaison A\* / IDA\* du §5 avec une heuristique **plus faible** (compter les tuiles mal placées) puis une **plus forte** (Manhattan + conflit linéaire). Le coût doit rester identique — l'optimalité ne dépend pas de la force de l'heuristique, seulement de son admissibilité. Ce qui change est le nombre d'expansions d'IDA\*, et c'est ce que vous mesurerez.


In [8]:
def mal_placees(state):
    # TODO etudiant : heuristique du nombre de tuiles mal placees (admissible).
    print("Exercice 3 a completer : heuristique 'tuiles mal placees'")
    return None


print("Exercice 3 a completer : heuristique 'tuiles mal placees', puis rejouer le tableau du §5")


Exercice 3 a completer : heuristique 'tuiles mal placees', puis rejouer le tableau du §5
